# Machine Translation using Teacher forcing

## Translating from english to hindi

In [1]:
import numpy as np
import pandas as pd
import sklearn
import torch
from torch import nn
import nltk


In [2]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.mps.is_available() else "cpu"
)

## 1. EDA

In [3]:
dir_name = "indic_languages_corpus/bilingual/hi-en"

In [4]:
import os
english = None
with open(os.path.join(os.path.abspath(dir_name), "train.en")) as f:
    english = f.readlines()

In [5]:
with open(os.path.join(os.path.abspath(dir_name), "train.hi")) as f:
    hindi = f.readlines()

In [6]:
english[:3]

['And what is their Sigil?\n',
 'I do not want to die.\n',
 "It's the same country I think.\n"]

In [7]:
hindi[:3]

['और उनके Sigil क्या है?\n',
 'मैं मरना नहीं चाहता.\n',
 'यह मुझे लगता है कि एक ही देश है.\n']

In [8]:
# X_train = ['<sos> ' + line + ' <eos>' for line in english]
# y_train = ['<sos>' + line + '<eos>' for line in hindi]

In [139]:
os.chdir('bilingual/hi-en')

In [140]:
os.getcwd()

'/Users/deven/Developer/Machine_Learning_Algorithms/MIT/6_7960/weekly/week4/homeworks/indic_languages_corpus/bilingual/hi-en'

In [141]:
NUM_SENTENCES = 70000
# strip the input and output of extra unnecessary characters
# store all the cleaned input and output sentences into input_sentences[] and output_sentences[]
# tokenize the Hindi (target) sentences using the indicNLP libary class and add <sos> (start-of-sentence) and <eos> (end-of-sentence)

input_sentences = []
output_sentences = []

count = 0
for line in open(r'train.en', encoding="utf-8"):
    count += 1

    if count > NUM_SENTENCES:
        break

    input_sentence = line.rstrip().strip("\n").strip('-') #we strip the sentence of '\n' and '-'
    input_sentences.append(input_sentence) #store all input sentences in the input sentences list

count = 0

for line in open(r'train.hi'):
    count += 1

    if count > NUM_SENTENCES:
        break
    output_sentence =  line.rstrip().strip("\n").strip('-')
    from indicnlp.tokenize import indic_tokenize
    line = indic_tokenize.trivial_tokenize(output_sentence) #we tokenize the hindi sentences

    output_sentences.append(['<sos>'] + line + ['<eos>']) #append the start and end tags to the tokenised sentences
    #each tokenied sentence is stored as a list in output sentences
print(type(input_sentences[10]))
print(type(output_sentences[10]))

<class 'str'>
<class 'list'>


In [142]:
os.chdir('../../../')

In [143]:
os.getcwd()

'/Users/deven/Developer/Machine_Learning_Algorithms/MIT/6_7960/weekly/week4/homeworks'

In [13]:
output_sentences[:3]

[['<sos>', 'और', 'उनके', 'Sigil', 'क्या', 'है', '?', '<eos>'],
 ['<sos>', 'मैं', 'मरना', 'नहीं', 'चाहता', '.', '<eos>'],
 ['<sos>',
  'यह',
  'मुझे',
  'लगता',
  'है',
  'कि',
  'एक',
  'ही',
  'देश',
  'है',
  '.',
  '<eos>']]

In [144]:
from typing import List, Union, Any
import nltk
import numpy as np
import regex as re
import torch
import unicodedata

def preprocess_sentence(w: str) -> str:
    w = w.lower().strip()
    # Create space between a word and the punctuation following it
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)
    # Replace everything with space except (a-z, A-Z, ".", "?", "!", ",")
    w = re.sub(r"[^a-zA-Z?.!,¿]+", " ", w)
    return w.strip()

def tokenize(sentence: str) -> List[str]:
    """Preprocesses a raw text sentence and splits it into a list of words."""
    clean_sentence = preprocess_sentence(sentence)
    return nltk.word_tokenize(clean_sentence)


class Tokenizer:
    # Accept either a List of raw strings OR a List of already tokenized lists
    def __init__(self, corpus: Union[List[str], List[List[str]]]):
        # 1. Define standard special tokens
        self.pad = "<pad>"
        self.unk = "<unk>"
        self.sos = "<sos>"
        self.eos = "<eos>"

        special_tokens = [self.pad, self.unk, self.sos, self.eos]

        # 2. Extract tokens depending on whether the corpus is raw strings or pre-tokenized lists
        tokenized_corpus = []
        for item in corpus:
            if isinstance(item, list):
                # Already tokenized (e.g., your Hindi output_sentences)
                tokenized_corpus.extend(item)
            else:
                # Raw string (e.g., your English input_sentences)
                tokenized_corpus.extend(tokenize(item))

        # 3. Filter out special tokens if they already exist in the text data
        unique_words = np.unique(tokenized_corpus)
        clean_words = [w for w in unique_words if w not in special_tokens]

        # 4. Finalize the vocabulary mappings
        self.vocab = np.array(special_tokens + clean_words)
        self.tok_to_id = {u: i for i, u in enumerate(self.vocab)}
        self.id_to_token = np.array(self.vocab)

    def encode(self, s: Union[str, List[str]], add_special_tokens: bool = True) -> torch.Tensor:
        # If it's a string, use our internal tokenizer. If it's a list, use it directly.
        words = s if isinstance(s, list) else tokenize(s)

        unk_id = self.tok_to_id[self.unk]
        ids = [self.tok_to_id.get(w, unk_id) for w in words]

        # If it's a raw string (English), you might want to add SOS/EOS.
        # Note: Your Hindi corpus already has <sos> and <eos> added during your file read loop,
        # so for Hindi you can set add_special_tokens=False.
        if add_special_tokens:
            ids = [self.tok_to_id[self.sos]] + ids + [self.tok_to_id[self.eos]]

        return torch.tensor(ids, dtype=torch.long)

    def decode(self, toks: torch.Tensor, skip_special_tokens: bool = True) -> str:
        tok_list = toks.tolist()

        if skip_special_tokens:
            exclude = {
                self.tok_to_id[self.pad],
                self.tok_to_id[self.sos],
                self.tok_to_id[self.eos],
            }
            words = [
                self.id_to_token[idx] for idx in tok_list if idx not in exclude
            ]
        else:
            words = [self.id_to_token[idx] for idx in tok_list]

        return " ".join(words)

    def pad_examples(self, tok_list: List[torch.Tensor]) -> torch.Tensor:
        """Pads a list of 1D tensors to the right using your preferred explicit loop operations."""
        if not tok_list:
            return torch.empty((0, 0), dtype=torch.long)

        # 1. Determine maximum dimensions
        num_examples = len(tok_list)
        max_len = max(tok.size(0) for tok in tok_list)
        pad_id = self.tok_to_id[self.pad]

        # 2. Allocate base tensor filled entirely with pad_id
        padded_tokens = torch.full(
            (num_examples, max_len),
            fill_value=pad_id,
            dtype=tok_list[0].dtype,
            device=tok_list[0].device,
        )

        # 3. Copy each sequence into its respective row slot
        for i, tok in enumerate(tok_list):
            padded_tokens[i, : tok.size(0)] = tok

        return padded_tokens


In [145]:
# 1. Fit the tokenizers on your loaded corpora
en_tokenizer = Tokenizer(input_sentences)  # Processes strings
hi_tokenizer = Tokenizer(output_sentences) # Safely handles your pre-tokenized lists

# 2. Encode all items into tensor lists
# We skip adding special tokens for Hindi since your loop already put them in!
en_tensors = [en_tokenizer.encode(s, add_special_tokens=True) for s in input_sentences]
hi_tensors = [hi_tokenizer.encode(s, add_special_tokens=False) for s in output_sentences]

# 3. Pad using your explicit loop method
padded_english = en_tokenizer.pad_examples(en_tensors)
padded_hindi = hi_tokenizer.pad_examples(hi_tensors)

print("Padded English Matrix Shape:", padded_english.shape)
print("Padded Hindi Matrix Shape:", padded_hindi.shape)


Padded English Matrix Shape: torch.Size([70000, 72])
Padded Hindi Matrix Shape: torch.Size([70000, 69])


In [146]:
padded_english

tensor([[    2,   519, 17165,  ...,     0,     0,     0],
        [    2,  7494,  4472,  ...,     0,     0,     0],
        [    2,  8060, 13238,  ...,     0,     0,     0],
        ...,
        [    2,  6967, 13238,  ...,     0,     0,     0],
        [    2,  7494,  6933,  ...,     0,     0,     0],
        [    2,  7092,  5426,  ...,     0,     0,     0]])

In [147]:
padded_hindi

tensor([[    2,  6101,  5454,  ...,     0,     0,     0],
        [    2, 17078, 16281,  ...,     0,     0,     0],
        [    2, 17419, 16834,  ...,     0,     0,     0],
        ...,
        [    2, 19455, 15335,  ...,     0,     0,     0],
        [    2, 16834,  5448,  ...,     0,     0,     0],
        [    2,  5597,  9111,  ...,     0,     0,     0]])

69

## Model PreProcessing

In [66]:
# BUFFER_SIZE stores the number of training points
BUFFER_SIZE = len(padded_english)

# BATCH_SIZE is set to 64. Training and gradient descent happens in batches of 64
BATCH_SIZE = 64

# the number of batches in one epoch (also, the number of steps during training, when we go batch by batch)
steps_per_epoch = BUFFER_SIZE // BATCH_SIZE

# the length of the embedded vector
embedding_dim = 256

# no of GRUs
units = 1024

# getting the size of the input and output vocabularies.
vocab_inp_size = len(en_tokenizer.vocab) + 1
vocab_tar_size = len(hi_tokenizer.vocab) + 1


In [67]:
steps_per_epoch

1321

In [68]:
vocab_inp_size

18987

In [69]:
vocab_tar_size

3396

### Data Loading to modelm

In [151]:
from torch.utils.data import Dataset,DataLoader,TensorDataset,random_split

In [148]:
ds = TensorDataset(padded_english,padded_hindi)

In [149]:
len(ds)

70000

In [150]:
ds[0]

(tensor([    2,   519, 17165,  8033, 15656, 14015,     7,     3,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0]),
 tensor([    2,  6101,  5454,  1720,  7364, 22068,   440,     3,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     

In [158]:
size = len(ds)
train_size = int(.5*size)
val_size = int(.3*size)
test_size = int(.2*size)
generator = torch.Generator().manual_seed(42)

train_subset,val_subset,test_subset = random_split(ds,lengths=[train_size,val_size,test_size],generator=generator)

In [159]:
len(train_subset)

35000

In [163]:
train_loader = DataLoader(train_subset, batch_size=64, shuffle=True,num_workers=2)
val_loader   = DataLoader(val_subset,   batch_size=64, shuffle=False,num_workers=2)
test_loader  = DataLoader(test_subset,  batch_size=64, shuffle=False,num_workers=2)


In [164]:
next(iter(train_loader))

[tensor([[    2, 17346, 10153,  ...,     0,     0,     0],
         [    2, 15641,  6387,  ...,     0,     0,     0],
         [    2,  6967, 13410,  ...,     0,     0,     0],
         ...,
         [    2,  4472, 17566,  ...,     0,     0,     0],
         [    2, 17165, 17283,  ...,     0,     0,     0],
         [    2,  6967, 17436,  ...,     0,     0,     0]]),
 tensor([[    2, 17037, 11661,  ...,     0,     0,     0],
         [    2, 15824,     5,  ...,     0,     0,     0],
         [    2,  5613, 10315,  ...,     0,     0,     0],
         ...,
         [    2,  7364,  4730,  ...,     0,     0,     0],
         [    2,  5623, 16555,  ...,     0,     0,     0],
         [    2,  4803,  5623,  ...,     0,     0,     0]])]

In [165]:
next(iter(val_loader))

[tensor([[    2,   292, 10614,  ...,     0,     0,     0],
         [    2,     6,     6,  ...,     0,     0,     0],
         [    2,  7494, 17383,  ...,     0,     0,     0],
         ...,
         [    2,  2593,     6,  ...,     0,     0,     0],
         [    2, 17220,     5,  ...,     0,     0,     0],
         [    2,  7494,  4521,  ...,     0,     0,     0]]),
 tensor([[    2, 21659, 16969,  ...,     0,     0,     0],
         [    2,    17,    17,  ...,     0,     0,     0],
         [    2, 17078, 11103,  ...,     0,     0,     0],
         ...,
         [    2, 15409,    17,  ...,     0,     0,     0],
         [    2,    11, 14379,  ...,     0,     0,     0],
         [    2, 17078, 18996,  ...,     0,     0,     0]])]

In [166]:
next(iter(test_loader))

[tensor([[    2, 17566, 12375,  ...,     0,     0,     0],
         [    2, 10153,  9768,  ...,     0,     0,     0],
         [    2,   655, 15865,  ...,     0,     0,     0],
         ...,
         [    2, 10409,     5,  ...,     0,     0,     0],
         [    2, 17566,  4521,  ...,     0,     0,     0],
         [    2,   519,  6452,  ...,     0,     0,     0]]),
 tensor([[    2, 11124, 21540,  ...,     0,     0,     0],
         [    2,     6, 17022,  ...,     0,     0,     0],
         [    2, 20016,   382,  ...,     0,     0,     0],
         ...,
         [    2, 12221,    15,  ...,     0,     0,     0],
         [    2, 11124, 20265,  ...,     0,     0,     0],
         [    2,  6101, 15824,  ...,     0,     0,     0]])]

## Model Building

##
Linear Multi-Head Attention module with kernel feature maps.

This module implements Kernel Linear Attention (Katharopoulos et al., 2020),
which replaces the softmax operator with an explicit, non-negative feature map
$\phi(x) = \text{ELU}(x) + 1$. By leveraging the associative property of matrix
multiplication, attention computation achieves linear time and memory complexity
$\mathcal{O}(N \cdot d^2)$ with respect to sequence length $N$.

The module natively supports three operational execution modes:
1. **Non-Causal Global Attention**: Bidirectional attention for vision or
encoder architectures (e.g., ViT, BERT). Computes $Q (K^T V)$ in $\mathcal{O}(N)$
time.
2. **Causal Training Mode**: Autoregressive attention computed using
cumulative prefixes (`torch.cumsum`) in $\mathcal{O}(N)$ parallel time.
3. **Recurrent Inference Mode**: Constant-time $\mathcal{O}(1)$ autoregressive
decoding per step by maintaining a fixed-size state tuple $(S_t, z_t)$
independent of sequence length.

Attributes:
dim (int): Total input and output feature dimension ($D$).
num_heads (int): Number of parallel attention heads ($H$).
n_hidden (int): Feature dimensionality per head ($d_k = d_v$).
eps (float): Epsilon scalar added to the denominator for numerical stability.
    qkv_proj (nn.Linear): Unified projection layer mapping $D \to 3 \times H \times d$.
out_proj (nn.Linear): Output linear layer mapping concatenated heads $H \times d \to D$.




In [ ]:
class AttentionHead(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self,):